# MedRAG — Fine-tune BGE v3 (TRỘN 6 bộ data người-xây-dựng + hard neg embedding + eval BioASQ)

Thay `pqa_artificial` (máy sinh, gây negative transfer) bằng HỖN HỢP data do người xây dựng:
MedMCQA, MedQuAD, StackExchange-Bio, HealthcareMagic, NFCorpus, BioASQ.

**Chống leakage:** BioASQ vừa dùng train vừa là benchmark eval -> tách 2 phần RỜI NHAU
theo id câu hỏi. Chỉ phần train của BioASQ vào data huấn luyện; phần eval giữ riêng để đánh giá.

## CHUẨN BỊ
- **Accelerator**: GPU **T4 x2** (không P100). **Internet**: On. Không cần Add Input gì (data tự tải).
- Chạy Cell 3 (INSPECT) trước để xác nhận field từng bộ. Nếu bộ nào lỗi/0 cặp -> báo lại để sửa adapter.


In [8]:
# Cell 1 — GPU + cài
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True).stdout or "BẬT GPU T4!")
!pip install -q sentence-transformers==2.7.0 faiss-cpu 2>/dev/null
print("OK")


Tesla T4
Tesla T4

OK


In [9]:
# Cell 2 — Cấu hình
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
BASE_MODEL = "BAAI/bge-small-en-v1.5"
OUT_BEST   = "/kaggle/working/MedicalRetriever-v3"
CKPT_DIR   = "/kaggle/working/checkpoints_v3"
os.makedirs(CKPT_DIR, exist_ok=True)

CAP_PER_SOURCE = 40000     # cap mỗi nguồn để không bộ nào áp đảo
EPOCHS = 3; BATCH_SIZE = 64; MAX_SEQ_LEN = 256; LR = 1e-5   # lr thấp hơn (tránh phá BGE)
EVAL_STEPS = 1000; HARD_TOPK = 15; HARD_SKIPTOP = 3
BIOASQ_N_EVAL = 300        # số câu hỏi BioASQ GIỮ RIÊNG để đánh giá (không train)


In [10]:
# Cell 3 — INSPECT: xem cấu trúc thật từng bộ (chạy trước, xác nhận field)
from datasets import load_dataset, get_dataset_split_names
def peek(repo, cfg=None, n=1):
    try:
        sp = get_dataset_split_names(repo, cfg) if cfg else get_dataset_split_names(repo)
        ds = load_dataset(repo, cfg, split=f"{sp[0]}[:{n}]") if cfg else load_dataset(repo, split=f"{sp[0]}[:{n}]")
        print(f"\n### {repo} [{cfg}] splits={sp}")
        print("cols:", ds.column_names)
        print("mẫu:", {k:(str(v)[:80]) for k,v in ds[0].items()})
    except Exception as e:
        print(f"\n### {repo} [{cfg}] LỖI: {e}")

peek("openlifescienceai/medmcqa")
peek("lavita/MedQuAD")
peek("sentence-transformers/pubmedqa")
peek("lavita/ChatDoctor-HealthCareMagic-100k")
peek("BeIR/nfcorpus", "queries")
peek("rag-datasets/rag-mini-bioasq", "question-answer-passages")



### openlifescienceai/medmcqa [None] splits=['train', 'test', 'validation']
cols: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']
mẫu: {'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': '2', 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tu', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}

### lavita/MedQuAD [None] splits=['train']
cols: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']
mẫu: {'document_id': '0000559', 'document_source': 'GHR', 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair', 'ca

README.md: 0.00B [00:00, ?B/s]


### sentence-transformers/pubmedqa [None] LỖI: Config name is missing.
Please pick one among the available configs: ['triplet', 'triplet-20', 'triplet-all']
Example of usage:
	`load_dataset('sentence-transformers/pubmedqa', 'triplet')`

### lavita/ChatDoctor-HealthCareMagic-100k [None] splits=['train']
cols: ['instruction', 'input', 'output']
mẫu: {'instruction': "If you are a doctor, please answer the medical questions based on the patient's ", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting dow', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is'}

### BeIR/nfcorpus [queries] splits=['queries']
cols: ['_id', 'title', 'text']
mẫu: {'_id': 'PLAIN-3', 'title': '', 'text': 'Breast Cancer Cells Feed on Cholesterol'}

### rag-datasets/rag-mini-bioasq [question-answer-passages] splits=['test']
cols: ['question', 'answer', 'relevant_passage_ids', 'id']
mẫu: {'question': 'Is Hirschsprung disease a mendelian or a

In [11]:
# Cell 4 — Adapter + nạp từng bộ về (query, positive). Phòng thủ: thử nhiều tên field, skip lỗi.
from datasets import load_dataset, get_dataset_split_names
def g(row, *names):
    for n in names:
        v = row.get(n)
        if v not in (None, ""): return v
    return None
def first_split(repo, cfg=None):
    try: sp = get_dataset_split_names(repo, cfg) if cfg else get_dataset_split_names(repo)
    except Exception: sp=[]
    return load_dataset(repo, cfg, split=sp[0]) if cfg else load_dataset(repo, split=sp[0])

def load_medmcqa(cap):
    ds = first_split("openlifescienceai/medmcqa")
    out=[]
    for r in ds:
        q, exp = g(r,"question"), g(r,"exp","explanation")
        if q and exp: out.append((q,exp))
        if len(out)>=cap*2: break
    return out
def load_medquad(cap):
    ds = first_split("lavita/MedQuAD"); out=[]
    for r in ds:
        q,a = g(r,"question","Question"), g(r,"answer","Answer")
        if q and a: out.append((q,a))
        if len(out)>=cap*2: break
    return out
def load_st_pubmedqa(cap):
    # bản PubMedQA đã chuẩn hoá retrieval bởi sentence-transformers (parquet, không script)
    ds = first_split("sentence-transformers/pubmedqa"); out=[]
    for r in ds:
        q,a = g(r,"anchor","question","query"), g(r,"positive","answer","pos")
        if q and a: out.append((q,a))
        if len(out)>=cap*2: break
    return out
def load_healthmagic(cap):
    ds = first_split("lavita/ChatDoctor-HealthCareMagic-100k"); out=[]
    for r in ds:
        q,a = g(r,"input","instruction","question"), g(r,"output","answer","response")
        if q and a: out.append((q,a))
        if len(out)>=cap*2: break
    return out
def load_nfcorpus(cap):
    # BEIR: queries + corpus + qrels(train)
    q_ds = load_dataset("BeIR/nfcorpus","queries",split="queries")
    c_ds = load_dataset("BeIR/nfcorpus","corpus",split="corpus")
    qmap = {str(r["_id"]): r["text"] for r in q_ds}
    cmap = {str(r["_id"]): ((r.get("title") or "")+" "+(r.get("text") or "")).strip() for r in c_ds}
    try: qrels = load_dataset("BeIR/nfcorpus-qrels", split="train")
    except Exception: qrels = load_dataset("BeIR/nfcorpus-qrels", split=get_dataset_split_names("BeIR/nfcorpus-qrels")[0])
    out=[]
    for r in qrels:
        if int(r.get("score",0))<=0: continue
        qid, cid = str(r["query-id"]), str(r["corpus-id"])
        if qid in qmap and cid in cmap: out.append((qmap[qid], cmap[cid]))
        if len(out)>=cap*2: break
    return out

print("Adapter sẵn sàng (BioASQ xử lý riêng ở Cell 5 vì cần tách eval).")


Adapter sẵn sàng (BioASQ xử lý riêng ở Cell 5 vì cần tách eval).


In [12]:
# Cell 5 — BioASQ: tách train/eval RỜI NHAU (chống leakage) + tạo cặp train
import ast, random
def parse_ids(raw):
    if isinstance(raw,list): return [str(x) for x in raw]
    if isinstance(raw,str):
        try: return [str(x) for x in ast.literal_eval(raw.strip())]
        except Exception: return [t.strip() for t in raw.strip("[]").split(",") if t.strip()]
    return []
REPO="rag-datasets/rag-mini-bioasq"
def fs(name):
    try: sp=get_dataset_split_names(REPO,name)
    except Exception: sp=[]
    return load_dataset(REPO,name,split=sp[0] if sp else "train")
bio_qa = list(fs("question-answer-passages"))
bio_corpus_ds = fs("text-corpus")
bio_corpus = {}
for r in bio_corpus_ds:
    pid=str(r.get("id","")).strip(); p=r.get("passage")
    if pid and isinstance(p,str) and p.strip(): bio_corpus[pid]=p.strip()

rng=random.Random(42); idxs=list(range(len(bio_qa))); rng.shuffle(idxs)
eval_idx=set(idxs[:BIOASQ_N_EVAL])
bio_train=[bio_qa[i] for i in idxs[BIOASQ_N_EVAL:]]
bio_eval =[bio_qa[i] for i in idxs[:BIOASQ_N_EVAL]]

def load_bioasq_train(cap):
    out=[]
    for r in bio_train:
        q=(r.get("question") or "").strip()
        for pid in parse_ids(r.get("relevant_passage_ids")):
            if q and pid in bio_corpus: out.append((q,bio_corpus[pid]))
        if len(out)>=cap*2: break
    return out
print(f"BioASQ: {len(bio_train)} câu train / {len(bio_eval)} câu eval (rời nhau). corpus {len(bio_corpus)}")


BioASQ: 4419 câu train / 300 câu eval (rời nhau). corpus 40221


In [13]:
# Cell 6 — TRỘN: nạp tất cả, cap, làm sạch, dedup
import random
def clean_pair(q,p,min_len=15):
    if not q or not p: return None
    q,p=str(q).strip(),str(p).strip()
    if len(q)<5 or len(p)<min_len: return None
    return (q,p)
def mix(sources, cap, seed=42):
    rng=random.Random(seed); allp=[]; seen=set(); stats={}
    for name,pairs in sources.items():
        clean=[c for c in (clean_pair(q,p) for q,p in pairs) if c]
        if len(clean)>cap: clean=rng.sample(clean,cap)
        added=0
        for q,p in clean:
            if (q,p) not in seen:
                seen.add((q,p)); allp.append({"query":q,"positive":p,"source":name}); added+=1
        stats[name]=added
    rng.shuffle(allp); return allp, stats

sources={}
for name,fn in [("medmcqa",load_medmcqa),("medquad",load_medquad),("st_pubmedqa",load_st_pubmedqa),
                ("healthmagic",load_healthmagic),("nfcorpus",load_nfcorpus),("bioasq",load_bioasq_train)]:
    try:
        sources[name]=fn(CAP_PER_SOURCE); print(f"  {name}: {len(sources[name])} cặp thô")
    except Exception as e:
        print(f"  {name}: LỖI -> {e}"); sources[name]=[]

mixed, stats = mix(sources, CAP_PER_SOURCE)
print("\n=== SAU TRỘN (cap+clean+dedup) ===")
for k,v in stats.items(): print(f"  {k:12s}: {v}")
print(f"TỔNG: {len(mixed)} cặp")
queries_txt=[m["query"] for m in mixed]; positives_txt=[m["positive"] for m in mixed]


  medmcqa: 80000 cặp thô
  medquad: 16407 cặp thô
  st_pubmedqa: LỖI -> list index out of range
  healthmagic: 80000 cặp thô
  nfcorpus: 80000 cặp thô
  bioasq: 39798 cặp thô

=== SAU TRỘN (cap+clean+dedup) ===
  medmcqa     : 40000
  medquad     : 16358
  st_pubmedqa : 0
  healthmagic : 40000
  nfcorpus    : 39923
  bioasq      : 27676
TỔNG: 163957 cặp


In [14]:
# Cell 7 — MINE hard negative bằng BGE (lọc false-negative)
import numpy as np, faiss, torch, gc
from sentence_transformers import SentenceTransformer
miner=SentenceTransformer(BASE_MODEL, device="cuda")
ctx_emb=miner.encode(positives_txt,batch_size=256,show_progress_bar=True,convert_to_numpy=True).astype(np.float32)
q_emb=miner.encode(queries_txt,batch_size=256,show_progress_bar=True,convert_to_numpy=True).astype(np.float32)
cn=ctx_emb.copy(); faiss.normalize_L2(cn); idx=faiss.IndexFlatIP(cn.shape[1]); idx.add(cn)
qn=q_emb.copy(); faiss.normalize_L2(qn)
S,I=idx.search(qn, HARD_TOPK+HARD_SKIPTOP+2)
# điểm sim query-gold để so (lọc false negative: candidate sim >= gold*0.95 -> nghi relevant thật)
gold_sim=np.sum(qn*cn, axis=1)
FN_RATarget=0.95
hard={}
for i in range(len(queries_txt)):
    cnt=0
    for rank,j in enumerate(I[i]):
        j=int(j)
        if j==i or positives_txt[j]==positives_txt[i]: continue
        if S[i][rank] >= gold_sim[i]*FN_RATarget:   # quá giống gold -> nghi false negative -> bỏ
            continue
        cnt+=1
        if cnt<=HARD_SKIPTOP: continue
        hard[i]=j; break
print(f"Mined {len(hard)}/{len(queries_txt)} hard negative (đã lọc false-negative)")
del miner,ctx_emb,q_emb,cn,qn,idx; gc.collect(); torch.cuda.empty_cache()


model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/641 [00:00<?, ?it/s]

Batches:   0%|          | 0/641 [00:00<?, ?it/s]

Mined 47652/163957 hard negative (đã lọc false-negative)


In [15]:
# Cell 8 — InputExample + Evaluator BioASQ (dùng phần EVAL đã tách, chống leakage)
from sentence_transformers import InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
examples=[]
for i in range(len(queries_txt)):
    t=[queries_txt[i], positives_txt[i]]
    if i in hard: t.append(positives_txt[hard[i]])
    examples.append(InputExample(texts=t))
print(f"{len(examples)} InputExample ({len(hard)} có hard neg)")

# evaluator từ bio_eval (RỜI train) + corpus BioASQ
ir_corpus=dict(list(bio_corpus.items())[:15000]); cset=set(ir_corpus)
irq, irr={}, {}
for k,r in enumerate(bio_eval):
    q=(r.get("question") or "").strip(); rel=set(parse_ids(r.get("relevant_passage_ids")))&cset
    if q and rel: irq[f"q{k}"]=q; irr[f"q{k}"]=rel
evaluator=InformationRetrievalEvaluator(irq,ir_corpus,irr,name="bioasq_heldout",show_progress_bar=False,
    corpus_chunk_size=5000,mrr_at_k=[10],ndcg_at_k=[10],accuracy_at_k=[5,10],precision_recall_at_k=[5,10],map_at_k=[10])
print(f"Evaluator (held-out): {len(irq)} query | corpus {len(ir_corpus)}")


163957 InputExample (47652 có hard neg)
Evaluator (held-out): 177 query | corpus 15000


In [18]:
# Cell 9 — TRAIN từ BGE, lưu BEST theo evaluator held-out
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader
model=SentenceTransformer(BASE_MODEL, device="cuda"); model.max_seq_length=MAX_SEQ_LEN
loader=DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss=losses.MultipleNegativesRankingLoss(model)
warmup=int(len(loader)*EPOCHS*0.1)

print(f"{len(loader)} step/epoch × {EPOCHS} = {len(loader)*EPOCHS} step | lr={LR}")
model.fit(train_objectives=[(loader,loss)], evaluator=evaluator, epochs=EPOCHS, warmup_steps=warmup,
    optimizer_params={"lr":LR}, output_path=OUT_BEST, save_best_model=True, evaluation_steps=EVAL_STEPS,
    checkpoint_path=CKPT_DIR, checkpoint_save_steps=EVAL_STEPS, checkpoint_save_total_limit=2,
    use_amp=True, show_progress_bar=True)
print("✅ XONG. Best model:", OUT_BEST)

2562 step/epoch × 3 = 7686 step | lr=1e-05


Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

Iteration:   0%|          | 0/2562 [00:00<?, ?it/s]

Iteration:   0%|          | 0/2562 [00:00<?, ?it/s]

Iteration:   0%|          | 0/2562 [00:00<?, ?it/s]

✅ XONG. Best model: /kaggle/working/MedicalRetriever-v3


In [20]:
# Cell 10 — Đánh giá cuối trên BioASQ held-out: v3 vs baseline BGE
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
texts=list(bio_corpus.values())[:40000]; ids=list(bio_corpus.keys())[:40000]; idset=set(ids)
fq=[]
for r in bio_eval:
    q=(r.get("question") or "").strip(); rel=set(parse_ids(r.get("relevant_passage_ids")))&idset
    if q and rel: fq.append({"q":q,"rel":rel})
K=[5,10,20,50]
def ev(path):
    m=SentenceTransformer(path,device="cuda")
    ce=m.encode(texts,batch_size=256,show_progress_bar=True,convert_to_numpy=True).astype(np.float32); faiss.normalize_L2(ce)
    ix=faiss.IndexFlatIP(ce.shape[1]); ix.add(ce)
    qe=m.encode([x["q"] for x in fq],batch_size=256,show_progress_bar=True,convert_to_numpy=True).astype(np.float32); faiss.normalize_L2(qe)
    _,I=ix.search(qe,max(K)); out={}
    for k in K:
        out[f"recall@{k}"]=np.mean([len(set(ids[j] for j in I[i][:k])&fq[i]["rel"])/len(fq[i]["rel"]) for i in range(len(fq))])
    mrr=[]
    for i in range(len(fq)):
        rr=0
        for rank,j in enumerate(I[i][:max(K)],1):
            if ids[j] in fq[i]["rel"]: rr=1/rank; break
        mrr.append(rr)
    out["mrr"]=np.mean(mrr); return out
print(f"\nĐánh giá trên {len(fq)} câu BioASQ held-out")
rv=ev(OUT_BEST); rb=ev(BASE_MODEL)
print(f"\n{'metric':<12}{'v3':>10}{'baseline':>10}{'Δ':>10}")
for k in sorted(rv): print(f"{k:<12}{rv[k]:>10.4f}{rb[k]:>10.4f}{rv[k]-rb[k]:>+10.4f}")



Đánh giá trên 300 câu BioASQ held-out


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


metric              v3  baseline         Δ
mrr             0.7692    0.7740   -0.0048
recall@10       0.4724    0.4540   +0.0184
recall@20       0.5482    0.5197   +0.0285
recall@5        0.3805    0.3588   +0.0216
recall@50       0.6096    0.5890   +0.0206


In [23]:
# Cell 11 — Hybrid (v3 dense + BM25 RRF) trên BioASQ held-out
!pip install -q bm25s 2>/dev/null
import bm25s

texts_h = list(bio_corpus.values())[:40000]
ids_h = list(bio_corpus.keys())[:40000]
idset_h = set(ids_h)
fq_h = []
for r in bio_eval:
    q=(r.get("question") or "").strip(); rel=set(parse_ids(r.get("relevant_passage_ids")))&idset_h
    if q and rel: fq_h.append({"q":q,"rel":rel})

import numpy as np, faiss
from sentence_transformers import SentenceTransformer
K=[5,10,20,50]
m = SentenceTransformer(OUT_BEST, device="cuda")
ce = m.encode(texts_h, batch_size=256, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(ce); ix=faiss.IndexFlatIP(ce.shape[1]); ix.add(ce)
qe = m.encode([x["q"] for x in fq_h], batch_size=256, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(qe)
Dd, Id = ix.search(qe, max(K))

toks = bm25s.tokenize(texts_h, stopwords="en", show_progress=False)
bm = bm25s.BM25(); bm.index(toks, show_progress=False)

def rrf(a, b, k=60, topn=50):
    s={}
    for r,d in enumerate(a,1): s[d]=s.get(d,0)+1/(k+r)
    for r,d in enumerate(b,1): s[d]=s.get(d,0)+1/(k+r)
    return sorted(s, key=s.get, reverse=True)[:topn]

def metrics(all_ret):
    out={}
    for k in K:
        out[f"recall@{k}"]=np.mean([len(set(all_ret[i][:k])&fq_h[i]["rel"])/len(fq_h[i]["rel"]) for i in range(len(fq_h))])
    mrr=[]
    for i in range(len(fq_h)):
        rr=0
        for rank,d in enumerate(all_ret[i][:max(K)],1):
            if d in fq_h[i]["rel"]: rr=1/rank; break
        mrr.append(rr)
    out["mrr"]=np.mean(mrr); return out

dense_ret, hybrid_ret = [], []
for i in range(len(fq_h)):
    dids=[ids_h[j] for j in Id[i]]
    qt=bm25s.tokenize([fq_h[i]["q"]], stopwords="en", show_progress=False)
    res,_=bm.retrieve(qt, k=max(K), show_progress=False)
    bids=[ids_h[int(j)] for j in res[0]]
    dense_ret.append(dids)
    hybrid_ret.append(rrf(dids, bids))

rd=metrics(dense_ret); rh=metrics(hybrid_ret)
print(f"\n{'metric':<12}{'v3-dense':>10}{'v3-hybrid':>10}{'Δ':>10}")
for k in sorted(rd): print(f"{k:<12}{rd[k]:>10.4f}{rh[k]:>10.4f}{rh[k]-rd[k]:>+10.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.1 MB/s eta 0:00:00


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects



metric        v3-dense v3-hybrid         Δ
mrr             0.7692    0.8146   +0.0454
recall@10       0.4724    0.5173   +0.0449
recall@20       0.5482    0.5911   +0.0430
recall@5        0.3805    0.4100   +0.0295
recall@50       0.6096    0.6377   +0.0281
